Use Case: Azure Cloud Infrastructure Management Agent

Problem Statement

Managing cloud infrastructure involves repetitive, error-prone tasks:

Checking resource status across multiple services
Scaling resources up/down based on load
Diagnosing issues from logs and metrics
Enforcing compliance and cost optimization
Responding to incidents automatically
An agentic AI system can handle these tasks autonomously, escalating to humans only when needed.

Architecture

User Request / Alert
        ↓
  [Intent Classifier] → Determines action type
        ↓
  ┌─────────────────────────────────┐
  │  Tool-Using ReAct Agent         │
  │  ├── list_resources()           │
  │  ├── get_resource_status()      │
  │  ├── scale_resource()           │
  │  ├── check_metrics()            │
  │  ├── check_compliance()         │
  │  ├── estimate_cost()            │
  │  └── create_incident_ticket()   │
  └─────────────────────────────────┘
        ↓
  [Decision Engine] → Act or Escalate?
        ↓
  Action / Human Escalation
Built with: LangChain + LangGraph + Anthropic Claude

Note: Azure operations are simulated for demo purposes. Replace with real Azure SDK calls for production.

In [6]:
!pip install -q langchain langchain-openai langgraph python-dotenv

In [7]:
import json
from datetime import datetime, timedelta
import random
from dotenv import load_dotenv

load_dotenv()  # Loads ANTHROPIC_API_KEY from .env

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional

llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

In [8]:
# Simulated Azure environment state
AZURE_RESOURCES = {
    "vm-prod-web-01": {
        "type": "Virtual Machine",
        "resource_group": "rg-production",
        "location": "East US",
        "size": "Standard_D4s_v3",
        "status": "Running",
        "cpu_usage": 87.5,
        "memory_usage": 72.3,
        "os": "Ubuntu 22.04",
        "ip": "10.0.1.4",
        "monthly_cost": 140.16,
        "tags": {"env": "production", "team": "backend", "app": "web-api"}
    },
    "vm-prod-web-02": {
        "type": "Virtual Machine",
        "resource_group": "rg-production",
        "location": "East US",
        "size": "Standard_D2s_v3",
        "status": "Running",
        "cpu_usage": 45.2,
        "memory_usage": 58.1,
        "os": "Ubuntu 22.04",
        "ip": "10.0.1.5",
        "monthly_cost": 70.08,
        "tags": {"env": "production", "team": "backend", "app": "web-api"}
    },
    "vm-dev-test-01": {
        "type": "Virtual Machine",
        "resource_group": "rg-development",
        "location": "East US",
        "size": "Standard_B2ms",
        "status": "Running",
        "cpu_usage": 5.1,
        "memory_usage": 12.4,
        "os": "Ubuntu 22.04",
        "ip": "10.0.2.4",
        "monthly_cost": 60.74,
        "tags": {"env": "development", "team": "qa"}
    },
    "app-prod-api": {
        "type": "App Service",
        "resource_group": "rg-production",
        "location": "East US",
        "plan": "P2v3",
        "status": "Running",
        "cpu_usage": 65.0,
        "memory_usage": 70.2,
        "instances": 2,
        "monthly_cost": 220.00,
        "tags": {"env": "production", "team": "backend", "app": "main-api"}
    },
    "sql-prod-main": {
        "type": "Azure SQL Database",
        "resource_group": "rg-production",
        "location": "East US",
        "tier": "Standard S3",
        "status": "Online",
        "dtu_usage": 78.5,
        "storage_used_gb": 45.2,
        "storage_max_gb": 250,
        "monthly_cost": 150.00,
        "tags": {"env": "production", "team": "data", "app": "main-db"}
    },
    "storage-prod-data": {
        "type": "Storage Account",
        "resource_group": "rg-production",
        "location": "East US",
        "kind": "StorageV2",
        "status": "Available",
        "used_gb": 520.8,
        "monthly_cost": 12.50,
        "replication": "GRS",
        "tags": {"env": "production", "team": "data"}
    },
    "vm-staging-01": {
        "type": "Virtual Machine",
        "resource_group": "rg-staging",
        "location": "West US",
        "size": "Standard_D2s_v3",
        "status": "Stopped",
        "cpu_usage": 0,
        "memory_usage": 0,
        "os": "Windows Server 2022",
        "ip": "10.0.3.4",
        "monthly_cost": 0,
        "tags": {"env": "staging", "team": "frontend"}
    },
}

# Simulated alerts
ACTIVE_ALERTS = [
    {"id": "ALT-001", "severity": "Critical", "resource": "vm-prod-web-01", "message": "CPU usage exceeded 85% for 15 minutes", "time": "2024-01-15 14:30:00"},
    {"id": "ALT-002", "severity": "Warning", "resource": "sql-prod-main", "message": "DTU usage at 78%, approaching threshold", "time": "2024-01-15 14:25:00"},
    {"id": "ALT-003", "severity": "Info", "resource": "vm-dev-test-01", "message": "VM running with very low utilization (<10% CPU)", "time": "2024-01-15 13:00:00"},
]

# Compliance rules
COMPLIANCE_RULES = {
    "tagging": "All resources must have 'env' and 'team' tags",
    "encryption": "All storage accounts must use encryption at rest",
    "backup": "All production databases must have backup enabled",
    "region": "Production resources must be in East US or West US",
    "idle_vms": "VMs with <10% CPU for 24+ hours should be deallocated",
}

print(f"Azure Environment Loaded:")
print(f"  Resources: {len(AZURE_RESOURCES)}")
print(f"  Active Alerts: {len(ACTIVE_ALERTS)}")
print(f"  Compliance Rules: {len(COMPLIANCE_RULES)}")

Azure Environment Loaded:
  Resources: 7
  Active Alerts: 3
  Compliance Rules: 5


In [9]:
@tool
def list_resources(resource_group: str = "all", resource_type: str = "all") -> str:
    """List Azure resources, optionally filtered by resource group or type.
    resource_group: 'all', 'rg-production', 'rg-development', 'rg-staging'
    resource_type: 'all', 'Virtual Machine', 'App Service', 'Azure SQL Database', 'Storage Account'
    """
    results = []
    for name, info in AZURE_RESOURCES.items():
        if resource_group != "all" and info["resource_group"] != resource_group:
            continue
        if resource_type != "all" and info["type"] != resource_type:
            continue
        results.append({
            "name": name,
            "type": info["type"],
            "resource_group": info["resource_group"],
            "status": info["status"],
            "location": info["location"],
            "monthly_cost": info["monthly_cost"]
        })
    return json.dumps(results, indent=2)

@tool
def get_resource_status(resource_name: str) -> str:
    """Get detailed status and metrics for a specific Azure resource."""
    if resource_name not in AZURE_RESOURCES:
        return f"Resource '{resource_name}' not found."
    resource = AZURE_RESOURCES[resource_name]
    return json.dumps(resource, indent=2)

@tool
def check_metrics(resource_name: str, metric: str = "all") -> str:
    """Check performance metrics for a resource. 
    metric: 'cpu', 'memory', 'dtu', 'storage', or 'all'
    Returns current value and 24-hour trend.
    """
    if resource_name not in AZURE_RESOURCES:
        return f"Resource '{resource_name}' not found."
    
    resource = AZURE_RESOURCES[resource_name]
    metrics = {}
    
    if metric in ("cpu", "all") and "cpu_usage" in resource:
        current = resource["cpu_usage"]
        metrics["cpu"] = {
            "current": current,
            "avg_24h": round(current * 0.85, 1),
            "peak_24h": round(min(current * 1.15, 100), 1),
            "trend": "increasing" if current > 70 else "stable"
        }
    if metric in ("memory", "all") and "memory_usage" in resource:
        current = resource["memory_usage"]
        metrics["memory"] = {
            "current": current,
            "avg_24h": round(current * 0.9, 1),
            "peak_24h": round(min(current * 1.1, 100), 1),
            "trend": "stable"
        }
    if metric in ("dtu", "all") and "dtu_usage" in resource:
        current = resource["dtu_usage"]
        metrics["dtu"] = {
            "current": current,
            "avg_24h": round(current * 0.88, 1),
            "peak_24h": round(min(current * 1.12, 100), 1),
            "trend": "increasing" if current > 70 else "stable"
        }
    if metric in ("storage", "all") and "storage_used_gb" in resource:
        metrics["storage"] = {
            "used_gb": resource.get("storage_used_gb") or resource.get("used_gb"),
            "max_gb": resource.get("storage_max_gb", "N/A"),
            "usage_percent": round((resource.get("storage_used_gb", 0) / resource.get("storage_max_gb", 1)) * 100, 1) if "storage_max_gb" in resource else "N/A"
        }
    
    return json.dumps({"resource": resource_name, "metrics": metrics}, indent=2)

@tool
def scale_resource(resource_name: str, action: str, target_size: str = "") -> str:
    """Scale an Azure resource up/down or change instance count.
    action: 'scale_up', 'scale_down', 'add_instance', 'remove_instance', 'start', 'stop', 'deallocate'
    target_size: new VM size or tier (e.g., 'Standard_D8s_v3')
    """
    if resource_name not in AZURE_RESOURCES:
        return f"Resource '{resource_name}' not found."
    
    resource = AZURE_RESOURCES[resource_name]
    
    if action == "scale_up":
        old_size = resource.get("size") or resource.get("tier") or resource.get("plan")
        return json.dumps({
            "status": "success",
            "resource": resource_name,
            "action": "scale_up",
            "old_size": old_size,
            "new_size": target_size or "next tier up",
            "message": f"Resource {resource_name} scaled up from {old_size} to {target_size or 'next tier'}. Change will take effect in 2-5 minutes.",
            "estimated_new_cost": round(resource["monthly_cost"] * 1.8, 2)
        })
    elif action == "scale_down":
        old_size = resource.get("size") or resource.get("tier") or resource.get("plan")
        return json.dumps({
            "status": "success",
            "resource": resource_name,
            "action": "scale_down",
            "old_size": old_size,
            "new_size": target_size or "previous tier",
            "message": f"Resource {resource_name} scaled down. Change will take effect in 2-5 minutes.",
            "estimated_new_cost": round(resource["monthly_cost"] * 0.55, 2)
        })
    elif action == "stop":
        return json.dumps({
            "status": "success",
            "resource": resource_name,
            "action": "stopped",
            "message": f"Resource {resource_name} has been stopped. You will still be charged for the disk.",
            "estimated_savings": round(resource["monthly_cost"] * 0.7, 2)
        })
    elif action == "deallocate":
        return json.dumps({
            "status": "success",
            "resource": resource_name,
            "action": "deallocated",
            "message": f"Resource {resource_name} has been deallocated. Compute charges stopped.",
            "estimated_savings": round(resource["monthly_cost"], 2)
        })
    elif action == "start":
        return json.dumps({
            "status": "success",
            "resource": resource_name,
            "action": "started",
            "message": f"Resource {resource_name} is starting. It will be available in 1-3 minutes."
        })
    elif action == "add_instance":
        current = resource.get("instances", 1)
        return json.dumps({
            "status": "success",
            "resource": resource_name,
            "action": "add_instance",
            "old_instances": current,
            "new_instances": current + 1,
            "message": f"Added 1 instance. {resource_name} now has {current + 1} instances.",
            "estimated_new_cost": round(resource["monthly_cost"] * (current + 1) / current, 2)
        })
    
    return f"Unknown action: {action}"

@tool
def get_alerts(severity: str = "all") -> str:
    """Get active Azure Monitor alerts. severity: 'all', 'Critical', 'Warning', 'Info'"""
    alerts = ACTIVE_ALERTS if severity == "all" else [a for a in ACTIVE_ALERTS if a["severity"] == severity]
    return json.dumps(alerts, indent=2)

@tool
def check_compliance(resource_name: str = "all") -> str:
    """Check compliance status of resources against organizational policies."""
    issues = []
    resources_to_check = AZURE_RESOURCES if resource_name == "all" else {resource_name: AZURE_RESOURCES.get(resource_name, {})}
    
    for name, info in resources_to_check.items():
        if not info:
            continue
        tags = info.get("tags", {})
        if "env" not in tags or "team" not in tags:
            issues.append({"resource": name, "rule": "tagging", "issue": "Missing required tags (env, team)", "severity": "High"})
        if info["type"] == "Virtual Machine" and info.get("cpu_usage", 100) < 10 and info.get("status") == "Running":
            issues.append({"resource": name, "rule": "idle_vms", "issue": f"VM running with {info['cpu_usage']}% CPU — consider deallocating", "severity": "Medium"})
        if info["type"] == "Azure SQL Database" and info.get("tags", {}).get("env") == "production":
            pass  # backup assumed OK for demo
    
    summary = {
        "total_resources_checked": len(resources_to_check),
        "compliance_issues": len(issues),
        "issues": issues
    }
    return json.dumps(summary, indent=2)

@tool
def estimate_cost(resource_group: str = "all", period: str = "monthly") -> str:
    """Estimate costs for resources. period: 'monthly' or 'annual'"""
    multiplier = 12 if period == "annual" else 1
    costs = {}
    total = 0
    
    for name, info in AZURE_RESOURCES.items():
        if resource_group != "all" and info["resource_group"] != resource_group:
            continue
        rg = info["resource_group"]
        cost = info["monthly_cost"] * multiplier
        costs.setdefault(rg, {"resources": [], "subtotal": 0})
        costs[rg]["resources"].append({"name": name, "type": info["type"], "cost": round(cost, 2)})
        costs[rg]["subtotal"] = round(costs[rg]["subtotal"] + cost, 2)
        total += cost
    
    return json.dumps({"period": period, "total_cost": round(total, 2), "by_resource_group": costs}, indent=2)

@tool
def create_incident_ticket(title: str, severity: str, description: str, assigned_to: str = "oncall") -> str:
    """Create an incident ticket for tracking. severity: 'P1', 'P2', 'P3', 'P4'"""
    ticket = {
        "ticket_id": f"INC-{random.randint(10000, 99999)}",
        "title": title,
        "severity": severity,
        "description": description,
        "assigned_to": assigned_to,
        "status": "Open",
        "created_at": datetime.now().isoformat(),
        "message": "Incident ticket created successfully."
    }
    return json.dumps(ticket, indent=2)

print("Azure Management Tools defined:")
for t in [list_resources, get_resource_status, check_metrics, scale_resource, get_alerts, check_compliance, estimate_cost, create_incident_ticket]:
    print(f"  🔧 {t.name}")

Azure Management Tools defined:
  🔧 list_resources
  🔧 get_resource_status
  🔧 check_metrics
  🔧 scale_resource
  🔧 get_alerts
  🔧 check_compliance
  🔧 estimate_cost
  🔧 create_incident_ticket


In [10]:
AZURE_SYSTEM_PROMPT = """You are an Azure Cloud Infrastructure Management Agent. You help DevOps 
engineers and cloud administrators manage their Azure environment.

Your capabilities:
1. List and inspect Azure resources (VMs, App Services, Databases, Storage)
2. Monitor metrics (CPU, memory, DTU, storage)
3. Scale resources up/down based on usage patterns
4. Check and respond to active alerts
5. Verify compliance with organizational policies
6. Estimate and optimize costs
7. Create incident tickets for critical issues

Guidelines:
- Always check current metrics before recommending scaling actions
- For critical alerts, investigate the root cause before taking action
- When scaling, explain the cost implications
- Flag any compliance violations you discover
- Be proactive about cost optimization suggestions
- For destructive actions (deallocate, stop), explain the impact first
- Create incident tickets for P1/P2 issues automatically

You are operating in a SIMULATED environment for demo purposes."""

tools = [list_resources, get_resource_status, check_metrics, scale_resource, 
         get_alerts, check_compliance, estimate_cost, create_incident_ticket]

azure_agent = create_react_agent(llm, tools)

print("Azure Infrastructure Agent created with 8 tools!")

Azure Infrastructure Agent created with 8 tools!


/var/folders/3_/5vghw65x4qg_yqvp28pxrql00000gp/T/ipykernel_72170/4250919467.py:27: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  azure_agent = create_react_agent(llm, tools)


In [11]:
result = azure_agent.invoke({"messages": [
    ("system", AZURE_SYSTEM_PROMPT),
    ("human", """Please do a cost optimization review of our entire Azure infrastructure.
    Check for:
    1. Idle or underutilized resources
    2. Resources that could be right-sized
    3. Overall cost breakdown by resource group
    Give me a summary with specific savings recommendations.""")
]})

for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"\n🔧 [{role}] Tool calls:")
        for tc in msg.tool_calls:
            print(f"   → {tc['name']}({json.dumps(tc['args'])})")
    elif hasattr(msg, "content") and msg.content and role != "SystemMessage":
        content = msg.content if isinstance(msg.content, str) else str(msg.content)
        if len(content) > 20:
            print(f"\n💬 [{role}]: {content}")


💬 [HumanMessage]: Please do a cost optimization review of our entire Azure infrastructure.
    Check for:
    1. Idle or underutilized resources
    2. Resources that could be right-sized
    3. Overall cost breakdown by resource group
    Give me a summary with specific savings recommendations.

🔧 [AIMessage] Tool calls:
   → list_resources({"resource_group": "all"})
   → estimate_cost({"period": "monthly"})

💬 [ToolMessage]: [
  {
    "name": "vm-prod-web-01",
    "type": "Virtual Machine",
    "resource_group": "rg-production",
    "status": "Running",
    "location": "East US",
    "monthly_cost": 140.16
  },
  {
    "name": "vm-prod-web-02",
    "type": "Virtual Machine",
    "resource_group": "rg-production",
    "status": "Running",
    "location": "East US",
    "monthly_cost": 70.08
  },
  {
    "name": "vm-dev-test-01",
    "type": "Virtual Machine",
    "resource_group": "rg-development",
    "status": "Running",
    "location": "East US",
    "monthly_cost": 60.74
  },
  {